In [ ]:
# ============================================================
# Conformal Prediction: Calibration Split + Nonconformity Scores
# ============================================================
#
# Goal: split the EXISTING test set into calib + test (the trained
# regression model itself is NOT retrained -- it stays exactly as it
# is in production). This gives us a calibration set to compute
# nonconformity scores (absolute residuals in log scale, matching the
# TransformedTargetRegressor's log1p/expm1 target transform) and a
# smaller held-out test set to verify empirical coverage.
#
# Stratification: by price, bucketed into quantile bins (price itself
# is continuous, so train_test_split's stratify= needs a categorical
# proxy -- pd.qcut bins it into quantile-based groups so calib/test
# get comparable price distributions).

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from pathlib import Path
import sys

root_dir = "../../"
sys.path.append(str(root_dir))

from config.config_script import (
    MODEL_REGRESSION_PATH,
    ARTIFACT_DIR,
    feature_cols,
)

# ------------------------------------------------------------
# 1. Load the existing test set (X_test, y_test) and the trained
#    regression pipeline. Adjust these loads to match wherever your
#    current X_test / y_test actually live (parquet paths from
#    config_script.py, e.g. X_TEST_REGRESSION_PATH / Y_TEST_REGRESSION_PATH).
# ------------------------------------------------------------

from config.config_script import X_TEST_REGRESSION_PATH, Y_TEST_REGRESSION_PATH

X_test_full = pd.read_parquet(X_TEST_REGRESSION_PATH)
y_test_full = pd.read_parquet(Y_TEST_REGRESSION_PATH).squeeze()  # squeeze -> Series if single column

regression_pipeline = joblib.load(MODEL_REGRESSION_PATH)

print(f"Existing test set size: {len(X_test_full)}")

# ------------------------------------------------------------
# 2. Stratified split: existing test set -> calib + new (smaller) test
#
# NOTE ON SPLIT SIZE: no fixed convention here -- 50/50 is a
# reasonable default that keeps both the calibration set and the
# final held-out test set large enough to be stable, given the
# already-small eligible dataset. Adjust calib_size if either half
# ends up too small to trust (rule of thumb: conformal calibration
# quantiles get noisy below roughly 50-100 points).
# ------------------------------------------------------------

CALIB_SIZE = 0.5
RANDOM_STATE = 42
N_PRICE_BINS = 5  # quintiles; reduce if any bin ends up too small to stratify

price_bins = pd.qcut(y_test_full, q=N_PRICE_BINS, duplicates="drop")
print(f"\nPrice bin distribution in test set:")
print(price_bins.value_counts().sort_index())

X_calib, X_test_new, y_calib, y_test_new = train_test_split(
    X_test_full,
    y_test_full,
    test_size=(1 - CALIB_SIZE),
    stratify=price_bins,
    random_state=RANDOM_STATE,
)

print(f"\nCalibration set size: {len(X_calib)}")
print(f"New (smaller) test set size: {len(X_test_new)}")

# ------------------------------------------------------------
# 3. Compute nonconformity scores on the calibration set.
#
# Nonconformity score = |log1p(y_true) - log1p(y_pred)|
#
# Computed in log scale to match the TransformedTargetRegressor's
# internal transform (log1p/expm1) -- residuals should be measured on
# the scale the model actually optimizes/errs on, not raw dollars
# (a $5 error on a $10 item is very different from a $5 error on a
# $500 item; log scale makes this comparable).
# ------------------------------------------------------------

y_calib_pred = regression_pipeline.predict(X_calib[feature_cols])

# Ensure y_calib and y_calib_pred are 1D numpy arrays before applying np.log1p
y_calib_np = y_calib.values.ravel() if isinstance(y_calib, pd.Series) else y_calib.ravel()
y_calib_pred_np = y_calib_pred.ravel()

nonconformity_scores = np.abs(np.log1p(y_calib_np) - np.log1p(y_calib_pred_np))

print(f"\nNonconformity scores (calibration set):")
print(f"  count: {len(nonconformity_scores)}")
print(f"  mean : {nonconformity_scores.mean():.4f}")
print(f"  std  : {nonconformity_scores.std():.4f}")
print(f"  min  : {nonconformity_scores.min():.4f}")
print(f"  max  : {nonconformity_scores.max():.4f}")

# ------------------------------------------------------------
# 4. Compute the 90% conformal quantile.
#
# Split conformal prediction uses the ceil((n+1)*(1-alpha))/n
# quantile of the calibration nonconformity scores (NOT simply the
# 90th percentile) -- this small-sample correction is what gives
# split conformal its finite-sample coverage guarantee.
# ------------------------------------------------------------

ALPHA = 0.05  # 1 - 0.90 confidence level
n_calib = len(nonconformity_scores)
quantile_level = np.ceil((n_calib + 1) * (1 - ALPHA)) / n_calib
quantile_level = min(quantile_level, 1.0)  # guard against n_calib too small

conformal_quantile = np.quantile(nonconformity_scores, quantile_level)

print(f"\n90% conformal quantile (log scale): {conformal_quantile:.4f}")
print(f"Quantile level used: {quantile_level:.4f} (n_calib={n_calib})")

# ------------------------------------------------------------
# 5. Verify empirical coverage on the NEW (smaller) test set.
#
# For each test point: predicted interval = [expm1(log1p(pred) - q),
# expm1(log1p(pred) + q)]. Coverage = fraction of true prices that
# fall inside their own predicted interval.
# ------------------------------------------------------------

y_test_pred = regression_pipeline.predict(X_test_new[feature_cols]).ravel()
y_test_new_np = y_test_new.to_numpy().ravel() if hasattr(y_test_new, "to_numpy") else np.asarray(y_test_new).ravel()

log_pred = np.log1p(y_test_pred)
lower_log = log_pred - conformal_quantile
upper_log = log_pred + conformal_quantile

lower = np.expm1(lower_log)
upper = np.expm1(upper_log)

covered = (y_test_new_np >= lower) & (y_test_new_np <= upper)
empirical_coverage = covered.mean()

print(f"\nEmpirical coverage on held-out test set (target 90%): {empirical_coverage:.4f}")
print(f"Average interval width (dollars): {(upper - lower).mean():.2f}")
print(f"Median interval width (dollars): {np.median(upper - lower):.2f}")

# ------------------------------------------------------------
# 6. Save the conformal quantile as a serving artifact.
#
# This is the ONLY number the serving endpoint needs at inference
# time -- not the full calibration set, not the residual array. Given
# a new point prediction, the interval is just
# [expm1(log1p(pred) - q), expm1(log1p(pred) + q)].
# ------------------------------------------------------------

CONFORMAL_ARTIFACT_PATH = ARTIFACT_DIR / "models" / "conformal_quantile_90.joblib"

conformal_artifact = {
    "quantile": float(conformal_quantile),
    "alpha": ALPHA,
    "confidence_level": 1 - ALPHA,
    "n_calib": n_calib,
    "empirical_coverage_on_test": float(empirical_coverage),
    "scale": "log1p",  # documents that `quantile` is in log1p space, not raw dollars
}

joblib.dump(conformal_artifact, CONFORMAL_ARTIFACT_PATH)
print(f"\nSaved conformal artifact to: {CONFORMAL_ARTIFACT_PATH}")
print(conformal_artifact)

c:\Users\Alvin\miniconda3\envs\mlflow_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Existing test set size: 305

Price bin distribution in test set:
price
(4.999, 12.35]      61
(12.35, 16.056]     61
(16.056, 22.29]     61
(22.29, 73.464]     61
(73.464, 434.95]    61
Name: count, dtype: int64

Calibration set size: 152
New (smaller) test set size: 153

Nonconformity scores (calibration set):
  count: 152
  mean : 0.2701
  std  : 0.3330
  min  : 0.0001
  max  : 2.0578

90% conformal quantile (log scale): 0.6256
Quantile level used: 0.9079 (n_calib=152)

Empirical coverage on held-out test set (target 90%): 0.8693
Average interval width (dollars): 52.38
Median interval width (dollars): 24.70

Saved conformal artifact to: C:\Users\Alvin\Music\project_1\src\artifacts\models\conformal_quantile_90.joblib
{'quantile': 0.6255720205687836, 'alpha': 0.1, 'confidence_level': 0.9, 'n_calib': 152, 'empirical_coverage_on_test': 0.869281045751634, 'scale': 'log1p'}


In [ ]:

# ============================================================
# DIAGNOSTIC: why is empirical coverage (51.7%) so far from target (90%)?
#
# Run this right after the calibration script, using the same
# X_calib, y_calib, X_test_new, y_test_new, regression_pipeline,
# conformal_quantile variables already in memory.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Compare nonconformity score distributions: calib vs test.
#    If conformal prediction's exchangeability assumption holds,
#    these two distributions should look statistically similar.
#    A big gap here is the smoking gun.
# ------------------------------------------------------------

y_test_new_pred = regression_pipeline.predict(X_test_new[feature_cols]).ravel()
test_nonconformity = np.abs(np.log1p(y_test_new_np) - np.log1p(y_test_new_pred))

print("=== Nonconformity score comparison ===")
print(f"{'':20s} {'calib':>10s} {'test_new':>10s}")
print(f"{'count':20s} {len(nonconformity_scores):>10d} {len(test_nonconformity):>10d}")
print(f"{'mean':20s} {nonconformity_scores.mean():>10.4f} {test_nonconformity.mean():>10.4f}")
print(f"{'std':20s} {nonconformity_scores.std():>10.4f} {test_nonconformity.std():>10.4f}")
print(f"{'median':20s} {np.median(nonconformity_scores):>10.4f} {np.median(test_nonconformity):>10.4f}")
print(f"{'90th pct':20s} {np.quantile(nonconformity_scores, 0.90):>10.4f} {np.quantile(test_nonconformity, 0.90):>10.4f}")
print(f"{'max':20s} {nonconformity_scores.max():>10.4f} {test_nonconformity.max():>10.4f}")

# If test_new's nonconformity scores are systematically LARGER than
# calib's, the calibration quantile will undercover on test -- this
# is the most likely explanation for 51.7% << 90%.

# ------------------------------------------------------------
# 2. Sanity check: is X_calib[feature_cols] / X_test_new[feature_cols]
#    actually getting the same preprocessing as during original
#    training? A silent column mismatch (e.g. wrong dtype, a column
#    the ColumnTransformer doesn't recognize, NaNs introduced by the
#    split) would inflate errors on whichever split index ordering
#    got scrambled.
# ------------------------------------------------------------

print("\n=== Feature sanity check ===")
print("X_calib dtypes:")
print(X_calib[feature_cols].dtypes)
print(f"\nX_calib NaN counts:\n{X_calib[feature_cols].isna().sum()[X_calib[feature_cols].isna().sum() > 0]}")
print(f"\nX_test_new NaN counts:\n{X_test_new[feature_cols].isna().sum()[X_test_new[feature_cols].isna().sum() > 0]}")

# ------------------------------------------------------------
# 3. Check: was the ORIGINAL X_test_full/y_test_full already used to
#    pick/validate the model (e.g. hyperparameter tuning, early
#    stopping decisions, or even just repeatedly "peeked at" during
#    iteration)? If so, splitting it into calib+test now doesn't
#    give a clean calibration set -- the model may have been
#    indirectly fit to patterns in this exact data already,
#    especially if any test-time information leaked into modeling
#    decisions across sessions.
# ------------------------------------------------------------

print("\n=== Prediction error breakdown (test_new), by price bin ===")
test_price_bins = pd.qcut(y_test_new_np, q=3, duplicates="drop")
comparison_df = pd.DataFrame({
    "y_true": y_test_new_np,
    "y_pred": y_test_new_pred,
    "abs_error_dollars": np.abs(y_test_new_np - y_test_new_pred),
    "nonconformity": test_nonconformity,
    "price_bin": test_price_bins,
})
print(comparison_df.groupby("price_bin", observed=True)[["abs_error_dollars", "nonconformity"]].agg(["mean", "median", "max"]))

# ------------------------------------------------------------
# 4. Direct check: what quantile level WOULD have given 90% coverage
#    on test_new, using test_new's own nonconformity scores? Compare
#    that to the quantile we actually computed from calib.
# ------------------------------------------------------------

ideal_quantile_for_test = np.quantile(test_nonconformity, 0.95)
print(f"\nQuantile actually used (from calib):     {conformal_quantile:.4f}")
print(f"Quantile that WOULD give 90% cov on test: {ideal_quantile_for_test:.4f}")
print(f"Ratio: {ideal_quantile_for_test / conformal_quantile:.2f}x")

=== Nonconformity score comparison ===
                          calib   test_new
count                       152        153
mean                     0.2701     0.3176
std                      0.3330     0.3732
median                   0.1579     0.1758
90th pct                 0.6155     0.7458
max                      2.0578     2.1762

=== Feature sanity check ===
X_calib dtypes:
title_length                            int64
title_word_count                        int64
volume_count                          float64
text_risk_score                         int64
volume_tier_encoded                     int64
condition_encoded                       int64
total_bonus_count                       int64
currency                               object
seller_location_grouped                object
is_boxset                                bool
is_special_edition                       bool
boxset_side_story_edition_included       bool
standalone_side_story_edition            bool
is_first_print  